In [ ]:
import ee
import datetime
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

countries = {
    "BBOX": ee.Geometry.BBox(-94.187, -39.020, 37.062, 18.229)

}

variables = [
    "temperature_2m",
    "skin_temperature",
    "soil_temperature_level_1",
    "soil_temperature_level_2",
    "soil_temperature_level_3",
    "volumetric_soil_water_layer_1",
    "volumetric_soil_water_layer_2",
    "volumetric_soil_water_layer_3",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "surface_pressure",
    "total_precipitation_sum",
    "surface_latent_heat_flux_sum",
    "surface_net_solar_radiation_sum",
    "evaporation_from_vegetation_transpiration_sum"
]

start_date = datetime.date(1990, 1, 1) 
end_date = datetime.date(2024, 12, 31)

for country_name, bbox in countries.items():
    current_date = start_date
    while current_date <= end_date:
        next_date = current_date + datetime.timedelta(days=1)
        date_str = current_date.strftime("%Y-%m-%d")
        next_date_str = next_date.strftime("%Y-%m-%d")

        dataset = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                   .filterBounds(bbox)
                   .filterDate(date_str, next_date_str)  # Fix: Use proper date range
                   .select(variables))

        image = dataset.mean().clip(bbox)

        task = ee.batch.Export.image.toDrive(
            image=image,
            description=f"ERA5_{country_name}_{date_str}",
            folder="GEE_Exports_BBOX",
            fileNamePrefix=f"ERA5_{country_name}_{date_str}",
            region=bbox,
            scale=10000,
            maxPixels=1e13
        )

        task.start()
        print(f"Exporting ERA5 data for {country_name}, {date_str}")

        current_date = next_date

print("All daily exports initiated!")